# Notebook 04 — Evaluation

This notebook implements the evaluation stage of the M-S-O pipeline:
1. Load all simulation results (3 policies × 20 replications)
2. Compute 95% confidence intervals for each policy-metric combination
3. Perform paired t-tests (CRN-paired) for all 3 policy pairs × 5 metrics
4. Compute Cohen's d effect sizes
5. Generate all 10 figures from the visualization plan
6. Save summary_stats.csv and pairwise_tests.csv

**Prerequisite:** Run notebooks 01–03 first.

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.utils.config_loader import load_config
from src.optimization.lp_formulation import LPDispatchSolver
from src.evaluation.report_builder import EvaluationEngine
from src.evaluation.statistics import PRIMARY_METRICS
from src.evaluation.visualizations import plot_all

print('Imports successful')

## 1. Load All Results

In [ ]:
results_dir = repo_root / 'data' / 'results'
results = {}
for policy in ['random', 'greedy', 'lp']:
    path = results_dir / f'sim_results_{policy}.csv'
    results[policy] = pd.read_csv(path)
    print(f'{policy}: {len(results[policy])} replications loaded')

# Load forecast data
lambda_hat_df = pd.read_csv(repo_root / 'data' / 'processed' / 'lambda_hat.csv', index_col=0)
lambda_hat = lambda_hat_df.values  # (12, 24)

lambda_true_df = pd.read_csv(repo_root / 'data' / 'synthetic' / 'lambda_true.csv')
# Extract test day (day=6)
test_day_df = lambda_true_df[lambda_true_df['day'] == 6].copy()
hour_cols = [c for c in test_day_df.columns if c.startswith('hour_')]
lambda_true_test = test_day_df[hour_cols].values  # shape (12, 24)

forecast_metrics_df = pd.read_csv(repo_root / 'data' / 'processed' / 'forecast_metrics.csv')

x_sol = LPDispatchSolver.load(repo_root / 'data' / 'processed' / 'lp_assignment.json')

print('\nAll inputs loaded.')

## 2. Summary Statistics with 95% Confidence Intervals

In [ ]:
engine = EvaluationEngine(results)
summary = engine.compute_summary()

print('Summary Statistics (mean ± 95% CI):')
print('=' * 80)
for metric in PRIMARY_METRICS:
    print(f'\n--- {metric} ---')
    sub = summary[summary['metric'] == metric][['policy', 'mean', 'std', 'ci_lower', 'ci_upper']]
    print(sub.to_string(index=False))

## 3. Pairwise Hypothesis Tests

In [ ]:
pairwise = engine.pairwise_tests()

print('Pairwise t-tests (H0: no difference between policies):')
print('=' * 80)
print(pairwise.to_string(index=False))

In [ ]:
# Count significant results
n_sig = pairwise['significant'].sum()
n_total = len(pairwise)
print(f'\nSignificant at alpha=0.05: {n_sig}/{n_total} comparisons')

# Show significant ones
sig_df = pairwise[pairwise['significant']]
if len(sig_df) > 0:
    print('\nSignificant differences:')
    print(sig_df[['comparison', 'metric', 't_stat', 'p_value', 'cohens_d', 'effect_size']].to_string(index=False))

## 4. Save Results

In [ ]:
engine.save(results_dir)
print(f'Summary stats saved to {results_dir / "summary_stats.csv"}')
print(f'Pairwise tests saved to {results_dir / "pairwise_tests.csv"}')

## 5. Generate All 10 Figures

In [ ]:
figures_dir = results_dir / 'figures'

figs = plot_all(
    lambda_hat=lambda_hat,
    lambda_true=lambda_true_test,
    forecast_metrics=forecast_metrics_df,
    summary=summary,
    pairwise=pairwise,
    results=results,
    x_sol=x_sol,
    out_dir=figures_dir,
)

print(f'Generated {len(figs)} figures in {figures_dir}')
import os
for f in sorted(os.listdir(figures_dir)):
    print(f'  {f}')

## 6. Display Key Results

In [ ]:
from IPython.display import Image

# Display Fig 3: Fulfillment Rate
fig3_path = figures_dir / 'fig3_fulfillment_rate.png'
if fig3_path.exists():
    display(Image(filename=str(fig3_path)))

In [ ]:
# Display Fig 5: Delivery Time Boxplot
fig5_path = figures_dir / 'fig5_delivery_time_boxplot.png'
if fig5_path.exists():
    display(Image(filename=str(fig5_path)))

In [ ]:
# Display Fig 7: LP Heatmap
fig7_path = figures_dir / 'fig7_lp_heatmap.png'
if fig7_path.exists():
    display(Image(filename=str(fig7_path)))

## Evaluation Complete

**Outputs saved:**
- `data/results/summary_stats.csv` — Mean, std, 95% CI per policy per metric
- `data/results/pairwise_tests.csv` — Paired t-tests, p-values, Cohen's d
- `data/results/figures/fig1_*.png` … `fig10_*.png` — All 10 figures

**Pipeline Complete:** M → O → S → Evaluation